# 11 OpenOximetry Grouped Logistic Regression Baseline

## Scope and Outputs

This notebook builds the main-dataset grouped logistic regression baseline on canonical OpenOximetry features. It uses paired grouped folds, pooled evaluation metrics, threshold sensitivity checks, coefficient influence summaries, and label-sensitivity comparisons.

Input: `outputs/openox_canonical_window_features.csv` (produced by notebook 10).
Outputs:
- `outputs/openox_logreg_baseline_summary.csv`
- `outputs/openox_logreg_split_diagnostics.csv`
- `outputs/openox_logreg_threshold_sweep.csv`
- `outputs/openox_logreg_coefficient_summary.csv`
- `outputs/openox_logreg_label_sensitivity.csv`

This is the terminal baseline stage for current OpenOximetry comparison work.

### Data Preparation

Load exported canonical features, validate schema requirements, cast feature/label types, and summarize prevalence for both label variants.

In [ ]:
import csv
from pathlib import Path

import numpy as np
import pandas as pd

FEATURE_COLUMN_NAMES = (
    "mean", "std", "minimum", "maximum", "value_range", "rms", "skewness", "kurtosis", "energy",
    "derivative_std", "dominant_frequency_hz", "bandpower_0_5_2_hz", "bandpower_2_5_hz", "bandpower_ratio_0_5_2_to_2_5",
)

LABEL_VARIANTS = ("label", "label_min_spo2")

REQUIRED_COLUMNS = {
    "recording_id", "label", "label_min_spo2", *FEATURE_COLUMN_NAMES,
}


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks").is_dir() and (candidate / "outputs").is_dir():
            return candidate
    raise RuntimeError("Could not find project root containing notebooks/ and outputs/")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
FEATURE_CSV = PROJECT_ROOT / "outputs" / "openox_canonical_window_features.csv"

with FEATURE_CSV.open(newline="", encoding="utf-8") as handle:
    feature_rows = list(csv.DictReader(handle))

feature_df = pd.DataFrame(feature_rows)
missing = sorted(REQUIRED_COLUMNS.difference(feature_df.columns))
assert not missing, f"Missing required columns: {missing}"

for col in FEATURE_COLUMN_NAMES:
    feature_df[col] = pd.to_numeric(feature_df[col], errors="raise")
for col in LABEL_VARIANTS:
    feature_df[col] = pd.to_numeric(feature_df[col], errors="raise").astype(int)
feature_df["recording_id"] = feature_df["recording_id"].astype(str)

print(f"Feature source: {FEATURE_CSV}")
print(f"Rows: {len(feature_df)}")
print(f"Recordings: {feature_df['recording_id'].nunique()}")

for label_col in LABEL_VARIANTS:
    positive_rows = int(feature_df[label_col].sum())
    total_rows = int(len(feature_df))
    positive_recordings = int(feature_df.loc[feature_df[label_col] == 1, "recording_id"].nunique())
    total_recordings = int(feature_df["recording_id"].nunique())
    print(
        f"{label_col}: positive_rows={positive_rows}/{total_rows} "
        f"({positive_rows/total_rows:.4%}), positive_recordings={positive_recordings}/{total_recordings}"
    )

disagreements = int((feature_df["label"] != feature_df["label_min_spo2"]).sum())
print(f"Label disagreements (label vs label_min_spo2): {disagreements}")

#### Step 1 Complete: Feature Table Loaded and Validated
The previous code cell loads canonical OpenOximetry features, checks required columns, enforces numeric casting, and reports class prevalence and label disagreement counts.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix


def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    """Generate and display a confusion matrix heatmap."""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("True Label")
    ax.set_xlabel("Predicted Label")
    plt.tight_layout()
    return fig, ax


def print_classification_report(y_true, y_pred, label_name, variant_name):
    """Print formatted classification report."""
    print(f"\n{'-' * 80}")
    print(f"Classification Report: {label_name} (variant={variant_name})")
    print(f"{'-' * 80}")
    print(classification_report(y_true, y_pred, target_names=["Negative", "Positive"], digits=4))

#### Step 2 Complete: Reporting Utilities Defined
The previous code cell defines confusion-matrix plotting and formatted classification-report utilities used in later evaluation sections.

### Grouped Split Strategy

Define grouped-fold construction and split diagnostics so train/test partitions remain recording-disjoint while preserving full grouped coverage across folds.

In [ ]:
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold

seed = 42
outer_n_splits = 5
fixed_sweep_thresholds = (0.2, 0.3, 0.4, 0.5, 0.6)


def build_grouped_coverage_splits(stratify_target: np.ndarray, groups: np.ndarray, max_n_splits: int, seed: int):
    unique_groups = np.unique(groups)
    placeholder = np.zeros(len(groups), dtype=float)

    for requested_splits in range(max_n_splits, 1, -1):
        if len(unique_groups) < requested_splits:
            continue

        candidate_splitters = [("GroupKFold", GroupKFold(n_splits=requested_splits), [None])]
        candidate_splitters.append(
            (
                "StratifiedGroupKFold",
                None,
                list(range(seed, seed + 200)),
            )
        )

        for splitter_name, splitter, seed_candidates in candidate_splitters:
            for candidate_seed in seed_candidates:
                if splitter_name == "GroupKFold":
                    splits = [(tr_idx, te_idx) for tr_idx, te_idx in splitter.split(placeholder, stratify_target, groups)]
                else:
                    splitter = StratifiedGroupKFold(n_splits=requested_splits, shuffle=True, random_state=candidate_seed)
                    splits = [(tr_idx, te_idx) for tr_idx, te_idx in splitter.split(placeholder, stratify_target, groups)]

                if len(splits) != requested_splits:
                    continue

                if any(set(groups[tr_idx]).intersection(set(groups[te_idx])) for tr_idx, te_idx in splits):
                    continue

                test_group_sequence = np.concatenate([groups[te_idx] for _, te_idx in splits])
                if len(np.unique(test_group_sequence)) != len(unique_groups):
                    continue

                return splits, candidate_seed if candidate_seed is not None else seed, requested_splits, splitter_name

    raise RuntimeError("Could not find a grouped coverage split with full recording coverage")


def summarize_split_design(y: np.ndarray, groups: np.ndarray, splits: list[tuple[np.ndarray, np.ndarray]], label_name: str):
    rows = []
    for i, (tr_idx, te_idx) in enumerate(splits, start=1):
        tr_groups = groups[tr_idx]
        te_groups = groups[te_idx]
        rows.append(
            {
                "label_variant": label_name,
                "split_index": i,
                "train_group_count": len(set(tr_groups.tolist())),
                "test_group_count": len(set(te_groups.tolist())),
                "train_row_count": int(len(tr_idx)),
                "test_row_count": int(len(te_idx)),
                "train_class_count": int(len(np.unique(y[tr_idx]))),
                "test_class_count": int(len(np.unique(y[te_idx]))),
                "train_positive_rows": int(y[tr_idx].sum()),
                "test_positive_rows": int(y[te_idx].sum()),
                "train_positive_recordings": len(set(tr_groups[y[tr_idx] == 1].tolist())),
                "test_positive_recordings": len(set(te_groups[y[te_idx] == 1].tolist())),
                "group_overlap_count": len(set(tr_groups.tolist()).intersection(set(te_groups.tolist()))),
            }
        )
    return pd.DataFrame(rows)

#### Step 3 Complete: Grouped Split Builders Added
The previous code cell adds grouped-coverage split search and fold-level split diagnostics used by both label variants.

#### Step 4: Modeling and Analysis Function Definitions
Define the core training/evaluation loop, threshold-sweep builder, and coefficient-stability summarization utilities.

##### Step 4.1: Grouped Logistic Regression Runner
Implement grouped-fold training with standardization, constant-fold fallback for single-class folds, per-fold metrics, and pooled metrics.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.preprocessing import StandardScaler


def run_grouped_logreg(y: np.ndarray, groups: np.ndarray, splits: list[tuple[np.ndarray, np.ndarray]], label_name: str, split_seed: int, split_strategy: str, actual_fold_count: int):
    split_rows = []
    pooled_true = []
    pooled_pred = []
    fold_predictions = []

    for split_index, (tr_idx, te_idx) in enumerate(splits, start=1):
        x_train_raw, x_test_raw = X[tr_idx], X[te_idx]
        y_train, y_test = y[tr_idx], y[te_idx]
        train_class_count = int(len(np.unique(y_train)))

        scaler = StandardScaler()
        x_train = scaler.fit_transform(x_train_raw)
        x_test = scaler.transform(x_test_raw)

        if train_class_count < 2:
            model_type = "constant"
            constant_class = int(y_train[0])
            preds = np.full(len(y_test), constant_class, dtype=int)
            probs = preds.astype(float)
        else:
            model_type = "logreg"
            model = LogisticRegression(class_weight="balanced", max_iter=1000, solver="liblinear", random_state=seed)
            model.fit(x_train, y_train)
            probs = model.predict_proba(x_test)[:, 1]
            preds = (probs >= 0.5).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_test, preds, labels=[0, 1]).ravel()
        fold_predictions.append({"split_index": split_index, "y_true": y_test.copy(), "y_pred": preds.copy()})

        split_rows.append(
            {
                "label_variant": label_name,
                "split_index": split_index,
                "decision_threshold": 0.5,
                "model_type": model_type,
                "train_row_count": int(len(tr_idx)),
                "test_row_count": int(len(te_idx)),
                "train_group_count": int(len(set(groups[tr_idx].tolist()))),
                "test_group_count": int(len(set(groups[te_idx].tolist()))),
                "train_class_count": train_class_count,
                "single_class_train_fold": bool(train_class_count < 2),
                "accuracy": float(accuracy_score(y_test, preds)),
                "precision": float(precision_score(y_test, preds, zero_division=0)),
                "recall": float(recall_score(y_test, preds, zero_division=0)),
                "f1": float(f1_score(y_test, preds, zero_division=0)),
                "true_negative": int(tn),
                "false_positive": int(fp),
                "false_negative": int(fn),
                "true_positive": int(tp),
            }
        )
        pooled_true.extend(y_test.tolist())
        pooled_pred.extend(preds.tolist())

    split_df = pd.DataFrame(split_rows)
    split_design_df = summarize_split_design(y, groups, splits, label_name)
    pooled_tn, pooled_fp, pooled_fn, pooled_tp = confusion_matrix(np.asarray(pooled_true), np.asarray(pooled_pred), labels=[0, 1]).ravel()
    summary_row = {
        "label_variant": label_name,
        "split_strategy": split_strategy,
        "split_seed": int(split_seed),
        "requested_fold_count": int(outer_n_splits),
        "fold_count": int(actual_fold_count),
        "single_class_train_folds": int((split_df["train_class_count"] < 2).sum()),
        "single_class_test_folds": int((split_design_df["test_class_count"] < 2).sum()),
        "constant_prediction_folds": int((split_df["model_type"] == "constant").sum()),
        "positive_rows": int(y.sum()),
        "total_rows": int(len(y)),
        "positive_row_rate": float(y.mean()),
        "positive_recordings": int(feature_df.loc[y == 1, "recording_id"].nunique()),
        "total_recordings": int(feature_df["recording_id"].nunique()),
        "positive_recording_rate": float(feature_df.loc[y == 1, "recording_id"].nunique() / feature_df["recording_id"].nunique()),
        "pooled_test_rows": int(len(pooled_true)),
        "pooled_test_recordings": int(len(np.unique(np.concatenate([groups[te_idx] for _, te_idx in splits])))),
        "pooled_true_negative": int(pooled_tn),
        "pooled_false_positive": int(pooled_fp),
        "pooled_false_negative": int(pooled_fn),
        "pooled_true_positive": int(pooled_tp),
        "pooled_accuracy": float(accuracy_score(np.asarray(pooled_true), np.asarray(pooled_pred))),
        "pooled_precision": float(precision_score(np.asarray(pooled_true), np.asarray(pooled_pred), zero_division=0)),
        "pooled_recall": float(recall_score(np.asarray(pooled_true), np.asarray(pooled_pred), zero_division=0)),
        "pooled_f1": float(f1_score(np.asarray(pooled_true), np.asarray(pooled_pred), zero_division=0)),
        "mean_split_accuracy": float(split_df["accuracy"].mean()),
        "mean_split_precision": float(split_df["precision"].mean()),
        "mean_split_recall": float(split_df["recall"].mean()),
        "mean_split_f1": float(split_df["f1"].mean()),
        "split_accuracy_std": float(split_df["accuracy"].std(ddof=0)),
        "split_precision_std": float(split_df["precision"].std(ddof=0)),
        "split_recall_std": float(split_df["recall"].std(ddof=0)),
        "split_f1_std": float(split_df["f1"].std(ddof=0)),
    }
    return split_df, split_design_df, summary_row, fold_predictions, np.asarray(pooled_true), np.asarray(pooled_pred)

##### Step 4.2: Threshold-Sweep Evaluator
Define pooled threshold evaluation across fixed decision cutoffs for the primary label baseline.

In [ ]:
def build_threshold_sweep(y: np.ndarray, splits: list[tuple[np.ndarray, np.ndarray]]):
    rows = []
    threshold_predictions = {}
    for threshold in fixed_sweep_thresholds:
        pooled_true = []
        pooled_pred = []

        for tr_idx, te_idx in splits:
            scaler = StandardScaler()
            x_train = scaler.fit_transform(X[tr_idx])
            x_test = scaler.transform(X[te_idx])
            y_train = y[tr_idx]
            y_test = y[te_idx]

            if len(np.unique(y_train)) < 2:
                preds = np.full(len(y_test), int(y_train[0]), dtype=int)
            else:
                model = LogisticRegression(class_weight="balanced", max_iter=1000, solver="liblinear", random_state=seed)
                model.fit(x_train, y_train)
                probs = model.predict_proba(x_test)[:, 1]
                preds = (probs >= threshold).astype(int)
            pooled_true.extend(y_test.tolist())
            pooled_pred.extend(preds.tolist())

        pooled_true_arr = np.asarray(pooled_true)
        pooled_pred_arr = np.asarray(pooled_pred)
        tn, fp, fn, tp = confusion_matrix(pooled_true_arr, pooled_pred_arr, labels=[0, 1]).ravel()
        rows.append(
            {
                "threshold": float(threshold),
                "fold_count": int(len(splits)),
                "pooled_test_rows": int(len(pooled_true_arr)),
                "pooled_true_negative": int(tn),
                "pooled_false_positive": int(fp),
                "pooled_false_negative": int(fn),
                "pooled_true_positive": int(tp),
                "pooled_accuracy": float(accuracy_score(pooled_true_arr, pooled_pred_arr)),
                "pooled_precision": float(precision_score(pooled_true_arr, pooled_pred_arr, zero_division=0)),
                "pooled_recall": float(recall_score(pooled_true_arr, pooled_pred_arr, zero_division=0)),
                "pooled_f1": float(f1_score(pooled_true_arr, pooled_pred_arr, zero_division=0)),
            }
        )
        threshold_predictions[threshold] = (pooled_true_arr, pooled_pred_arr)

    return pd.DataFrame(rows).sort_values("threshold").reset_index(drop=True), threshold_predictions

##### Step 4.3: Coefficient Influence Summary
Define fold-wise coefficient extraction and aggregate stability metrics, including sign-consistency indicators and absolute-rank behavior.

In [ ]:
def summarize_coefficients(y: np.ndarray, splits: list[tuple[np.ndarray, np.ndarray]], label_name: str):
    coef_rows = []

    for split_index, (tr_idx, _) in enumerate(splits, start=1):
        scaler = StandardScaler()
        x_train = scaler.fit_transform(X[tr_idx])
        y_train = y[tr_idx]
        if len(np.unique(y_train)) < 2:
            continue
        model = LogisticRegression(class_weight="balanced", max_iter=1000, solver="liblinear", random_state=seed)
        model.fit(x_train, y_train)

        for feat, coef_val in zip(FEATURE_COLUMN_NAMES, model.coef_[0]):
            coef_rows.append(
                {
                    "label_variant": label_name,
                    "split_index": split_index,
                    "feature_name": feat,
                    "coefficient": float(coef_val),
                    "abs_coefficient": float(abs(coef_val)),
                }
            )

    coef_df = pd.DataFrame(coef_rows)
    coef_df["abs_rank"] = coef_df.groupby(["label_variant", "split_index"])["abs_coefficient"].rank(method="average", ascending=False)
    summary = (
        coef_df.groupby(["label_variant", "feature_name"], dropna=False)
        .agg(
            mean_coefficient=("coefficient", "mean"),
            std_coefficient=("coefficient", "std"),
            mean_abs_coefficient=("abs_coefficient", "mean"),
            std_abs_coefficient=("abs_coefficient", "std"),
            mean_abs_rank=("abs_rank", "mean"),
            std_abs_rank=("abs_rank", "std"),
            positive_sign_fraction=("coefficient", lambda values: float(np.mean(values > 0))),
            negative_sign_fraction=("coefficient", lambda values: float(np.mean(values < 0))),
            zero_sign_fraction=("coefficient", lambda values: float(np.mean(values == 0))),
            split_count=("split_index", "nunique"),
        )
        .reset_index()
    )
    summary["sign_consistency_rate"] = summary[["positive_sign_fraction", "negative_sign_fraction", "zero_sign_fraction"]].max(axis=1)
    summary = summary.sort_values(
        ["label_variant", "mean_abs_coefficient", "sign_consistency_rate", "mean_abs_rank"],
        ascending=[True, False, False, True],
    ).reset_index(drop=True)
    return summary

### Baseline Training and Aggregated Analysis

Run paired grouped training for both label variants, generate pooled threshold diagnostics, and summarize coefficient influence patterns across folds.

In [ ]:
X = feature_df.loc[:, FEATURE_COLUMN_NAMES].to_numpy(dtype=float)
groups = feature_df["recording_id"].to_numpy()

split_reference = feature_df.groupby("recording_id")["label"].transform("max").to_numpy(dtype=int)
y_primary = feature_df["label"].to_numpy(dtype=int)
y_secondary = feature_df["label_min_spo2"].to_numpy(dtype=int)

paired_splits, paired_split_seed, actual_fold_count, split_strategy = build_grouped_coverage_splits(split_reference, groups, outer_n_splits, seed)

primary_split_df, primary_design_df, primary_summary_row, primary_fold_preds, primary_pooled_true, primary_pooled_pred = run_grouped_logreg(y_primary, groups, paired_splits, "label", paired_split_seed, split_strategy, actual_fold_count)
secondary_split_df, secondary_design_df, secondary_summary_row, secondary_fold_preds, secondary_pooled_true, secondary_pooled_pred = run_grouped_logreg(y_secondary, groups, paired_splits, "label_min_spo2", paired_split_seed, split_strategy, actual_fold_count)

split_diagnostics_df = pd.concat(
    [
        primary_split_df.merge(primary_design_df, on=["label_variant", "split_index"], how="left"),
        secondary_split_df.merge(secondary_design_df, on=["label_variant", "split_index"], how="left"),
    ],
    ignore_index=True,
)

threshold_sweep_df, threshold_predictions = build_threshold_sweep(y_primary, paired_splits)

coefficient_summary_df = pd.concat(
    [
        summarize_coefficients(y_primary, paired_splits, "label"),
        summarize_coefficients(y_secondary, paired_splits, "label_min_spo2"),
    ],
    ignore_index=True,
)

baseline_summary_df = pd.DataFrame([primary_summary_row, secondary_summary_row])

label_sensitivity_df = pd.DataFrame(
    [
        {
            "label_variant": "label",
            "paired_fold_count": int(primary_summary_row["fold_count"]),
            "paired_test_rows": int(primary_summary_row["pooled_test_rows"]),
            "paired_test_recordings": int(primary_summary_row["pooled_test_recordings"]),
            "split_seed": int(paired_split_seed),
            "split_strategy": split_strategy,
            "positive_rows": int(primary_summary_row["positive_rows"]),
            "positive_row_rate": float(primary_summary_row["positive_row_rate"]),
            "positive_recordings": int(primary_summary_row["positive_recordings"]),
            "positive_recording_rate": float(primary_summary_row["positive_recording_rate"]),
            "pooled_accuracy": float(primary_summary_row["pooled_accuracy"]),
            "pooled_precision": float(primary_summary_row["pooled_precision"]),
            "pooled_recall": float(primary_summary_row["pooled_recall"]),
            "pooled_f1": float(primary_summary_row["pooled_f1"]),
        },
        {
            "label_variant": "label_min_spo2",
            "paired_fold_count": int(secondary_summary_row["fold_count"]),
            "paired_test_rows": int(secondary_summary_row["pooled_test_rows"]),
            "paired_test_recordings": int(secondary_summary_row["pooled_test_recordings"]),
            "split_seed": int(paired_split_seed),
            "split_strategy": split_strategy,
            "positive_rows": int(secondary_summary_row["positive_rows"]),
            "positive_row_rate": float(secondary_summary_row["positive_row_rate"]),
            "positive_recordings": int(secondary_summary_row["positive_recordings"]),
            "positive_recording_rate": float(secondary_summary_row["positive_recording_rate"]),
            "pooled_accuracy": float(secondary_summary_row["pooled_accuracy"]),
            "pooled_precision": float(secondary_summary_row["pooled_precision"]),
            "pooled_recall": float(secondary_summary_row["pooled_recall"]),
            "pooled_f1": float(secondary_summary_row["pooled_f1"]),
        },
        {
            "label_variant": "delta_label_min_minus_label",
            "paired_fold_count": int(primary_summary_row["fold_count"]),
            "paired_test_rows": int(primary_summary_row["pooled_test_rows"]),
            "paired_test_recordings": int(primary_summary_row["pooled_test_recordings"]),
            "split_seed": int(paired_split_seed),
            "split_strategy": split_strategy,
            "positive_rows": int(secondary_summary_row["positive_rows"] - primary_summary_row["positive_rows"]),
            "positive_row_rate": float(secondary_summary_row["positive_row_rate"] - primary_summary_row["positive_row_rate"]),
            "positive_recordings": int(secondary_summary_row["positive_recordings"] - primary_summary_row["positive_recordings"]),
            "positive_recording_rate": float(secondary_summary_row["positive_recording_rate"] - primary_summary_row["positive_recording_rate"]),
            "pooled_accuracy": float(secondary_summary_row["pooled_accuracy"] - primary_summary_row["pooled_accuracy"]),
            "pooled_precision": float(secondary_summary_row["pooled_precision"] - primary_summary_row["pooled_precision"]),
            "pooled_recall": float(secondary_summary_row["pooled_recall"] - primary_summary_row["pooled_recall"]),
            "pooled_f1": float(secondary_summary_row["pooled_f1"] - primary_summary_row["pooled_f1"]),
        },
    ]
)

print(f"Paired grouped coverage splits: {len(paired_splits)} folds (seed={paired_split_seed}, strategy={split_strategy})")
print("\nPooled baseline summary:")
print(baseline_summary_df.round(4).to_string(index=False))
print("\nSplit diagnostics (first 10 rows):")
print(split_diagnostics_df.head(10).to_string(index=False))
print("\nPrimary-label pooled threshold sweep:")
print(threshold_sweep_df.round(4).to_string(index=False))
print("\nCoefficient influence with stability checks (top 10 per label variant):")
for label_name in ["label", "label_min_spo2"]:
    print(f"\nTop features for {label_name}:")
    display_df = coefficient_summary_df[coefficient_summary_df["label_variant"] == label_name].head(10)
    print(display_df.round(4).to_string(index=False))
print("\nPaired label sensitivity:")
print(label_sensitivity_df.round(4).to_string(index=False))

#### Step 5 Complete: Paired Grouped Baselines Executed
The previous code cell runs grouped cross-validation for both label variants and materializes baseline, split, threshold, coefficient, and label-sensitivity tables.

### Per-Fold Classification Metrics

Inspect fold-level confusion matrices and classification reports for each label variant.

In [ ]:
print("=" * 80)
print("PER-FOLD CLASSIFICATION METRICS (Primary Label)")
print("=" * 80)

for fold_data in primary_fold_preds:
    split_idx = fold_data["split_index"]
    y_true = fold_data["y_true"]
    y_pred = fold_data["y_pred"]
    
    # Plot confusion matrix
    fig, ax = plot_confusion_matrix(y_true, y_pred, title=f"Fold {split_idx}: Confusion Matrix (label)")
    plt.show()
    
    # Print classification report
    print_classification_report(y_true, y_pred, f"Fold {split_idx}", "label")

print("\n" + "=" * 80)
print("PER-FOLD CLASSIFICATION METRICS (Secondary Label: label_min_spo2)")
print("=" * 80)

for fold_data in secondary_fold_preds:
    split_idx = fold_data["split_index"]
    y_true = fold_data["y_true"]
    y_pred = fold_data["y_pred"]
    
    # Plot confusion matrix
    fig, ax = plot_confusion_matrix(y_true, y_pred, title=f"Fold {split_idx}: Confusion Matrix (label_min_spo2)")
    plt.show()
    
    # Print classification report
    print_classification_report(y_true, y_pred, f"Fold {split_idx}", "label_min_spo2")

#### Step 6 Complete: Fold-Level Diagnostics Rendered
The previous code cell visualizes per-fold confusion matrices and prints detailed classification reports for both label variants.

### Pooled Test Set Classification Metrics

Inspect confusion matrices and classification reports aggregated across all held-out folds for each label variant.

In [ ]:
print("=" * 80)
print("POOLED TEST SET CLASSIFICATION METRICS")
print("=" * 80)

print("\n[1] PRIMARY LABEL (label) - Pooled across all folds")
fig, ax = plot_confusion_matrix(primary_pooled_true, primary_pooled_pred, title="Pooled: Confusion Matrix (label)")
plt.show()
print_classification_report(primary_pooled_true, primary_pooled_pred, "Pooled", "label")

print("\n[2] SECONDARY LABEL (label_min_spo2) - Pooled across all folds")
fig, ax = plot_confusion_matrix(secondary_pooled_true, secondary_pooled_pred, title="Pooled: Confusion Matrix (label_min_spo2)")
plt.show()
print_classification_report(secondary_pooled_true, secondary_pooled_pred, "Pooled", "label_min_spo2")

print("\n[3] LABEL SENSITIVITY COMPARISON")
print(f"Primary label positives:   {primary_pooled_true.sum()} / {len(primary_pooled_true)} ({primary_pooled_true.mean():.4%})")
print(f"Secondary label positives: {secondary_pooled_true.sum()} / {len(secondary_pooled_true)} ({secondary_pooled_true.mean():.4%})")
print(f"Label disagreements: {(primary_pooled_true != secondary_pooled_true).sum()}")

#### Step 7 Complete: Pooled Metrics Rendered
The previous code cell shows pooled confusion matrices and classification reports, plus primary-vs-secondary label disagreement context.

### Threshold-Sweep Classification Metrics

Inspect confusion matrices and classification reports across predefined decision thresholds for the primary label baseline.

In [ ]:
print("=" * 80)
print("THRESHOLD SWEEP CLASSIFICATION METRICS (Primary Label)")
print("=" * 80)

for threshold in sorted(threshold_predictions.keys()):
    y_true_sweep, y_pred_sweep = threshold_predictions[threshold]
    
    # Plot confusion matrix
    fig, ax = plot_confusion_matrix(y_true_sweep, y_pred_sweep, 
                                     title=f"Threshold {threshold:.1f}: Confusion Matrix (label)")
    plt.show()
    
    # Print classification report
    print_classification_report(y_true_sweep, y_pred_sweep, f"Threshold {threshold:.1f}", "label")

#### Step 8 Complete: Threshold Sensitivity Diagnostics Rendered
The previous code cell reports threshold-level confusion matrices and classification metrics for primary-label decision analysis.

### Artifact Export

Persist baseline summaries, split diagnostics, threshold sweep results, coefficient influence summaries, and label-sensitivity outputs to `outputs/`.

In [ ]:
baseline_summary_path = PROJECT_ROOT / "outputs" / "openox_logreg_baseline_summary.csv"
split_diag_path = PROJECT_ROOT / "outputs" / "openox_logreg_split_diagnostics.csv"
threshold_sweep_path = PROJECT_ROOT / "outputs" / "openox_logreg_threshold_sweep.csv"
coef_summary_path = PROJECT_ROOT / "outputs" / "openox_logreg_coefficient_summary.csv"
label_sensitivity_path = PROJECT_ROOT / "outputs" / "openox_logreg_label_sensitivity.csv"

baseline_summary_df.to_csv(baseline_summary_path, index=False)
split_diagnostics_df.to_csv(split_diag_path, index=False)
threshold_sweep_df.to_csv(threshold_sweep_path, index=False)
coefficient_summary_df.to_csv(coef_summary_path, index=False)
label_sensitivity_df.to_csv(label_sensitivity_path, index=False)

print(f"Wrote: {baseline_summary_path}")
print(f"Wrote: {split_diag_path}")
print(f"Wrote: {threshold_sweep_path}")
print(f"Wrote: {coef_summary_path}")
print(f"Wrote: {label_sensitivity_path}")

#### Step 9 Complete: Artifacts Written
The previous code cell writes all notebook 11 baseline artifacts to the project `outputs/` directory.